# Time, frequency, and WDM translation

Use one `transform` function for `L1Data`, covariance, and `BlockResult`.
Then let `Wheel` prepare a different representation for each block.

This walkthrough builds synthetic A/E TDI channels, chooses WDM divisions in two
ways, checks round trips and covariance weighting, and runs three stateless
blocks. The final blocks publish zero signals to demonstrate the exchange
contract; they do not fit a physical source model.

## Setup

Run these commands in a terminal **from the enchilada repository root**. They
assume your WDM checkout is the sibling directory `../wdm`, corresponding to
`~/Documents/lisa/wdm` in your setup.

```sh
uv sync --python 3.13 --extra examples
uv pip install ../wdm matplotlib
uv run --no-sync jupyter lab examples/domain_translation.ipynb
```

Select the kernel from that environment and run the cells from top to bottom.
WDM requires Python 3.13 or newer; Matplotlib is used only for this notebook's
plots. Install the unpublished WDM backend from your checkout. Install it after
project sync and launch with `--no-sync` so the environment retains it.

In [ ]:
import sys
from dataclasses import dataclass

import matplotlib.pyplot as plt
import numpy as np

from enchilada import (
    BlockResult,
    DataCovariance,
    L1Data,
    TranslatedCovariance,
    Wheel,
    transform,
)

if sys.version_info < (3, 13):
    raise RuntimeError("Choose a Python >=3.13 kernel for the local WDM backend.")

print(f"Kernel: {sys.executable}")

## 1. Create observations and their covariance

Use 128 samples with a two-second sampling interval. A non-unit interval makes
the normalization visible: enchilada stores Fourier coefficients as
`sample_interval_s * rfft(time_series)`.

Here the two channels have independent Gaussian noise with variance 0.04 per
time sample. This native time covariance will also be usable in other domains.

In [ ]:
rng = np.random.default_rng(7)
num_time_samples = 128
sample_interval_s = 2.0
sample_times_s = np.arange(num_time_samples) * sample_interval_s
channel_names = ("A", "E")
noise_standard_deviation = 0.2

injected_signal = {
    "A": np.sin(2 * np.pi * 0.035 * sample_times_s),
    "E": 0.6 * np.cos(2 * np.pi * 0.055 * sample_times_s + 0.4),
}
observed = L1Data(
    channel_data={
        name: values + rng.normal(0.0, noise_standard_deviation, num_time_samples)
        for name, values in injected_signal.items()
    },
    sample_rate_hz=1.0 / sample_interval_s,
    channel_names=channel_names,
    tdi_generation="2.0",
    physical_observable="strain",
    start_time_gps=100.0,
)
noise_covariance = DataCovariance.from_variance(
    reference_data=observed,
    time_sample_variance=noise_standard_deviation**2,
)
print(
    f"Domain: {observed.data_domain}; channel A shape: {observed.channel_data['A'].shape}"
)

## 2. Choose either WDM division count

The counts are linked by

```text
num_time_samples = num_frequency_divisions * num_time_divisions
```

Both counts must be positive even integers. Supply either one and the other is
derived. For our 128 samples, eight frequency divisions produce 16 time
divisions; eight time divisions produce 16 frequency divisions.

Each channel's WDM array has shape `(num_frequency_divisions + 1,
num_time_divisions)`. The extra row belongs to the Wilson basis's edge layout.
No padding or trimming is performed.

In [ ]:
spectrum = transform(observed, "frequency")
wdm_data = transform(observed, "wdm", num_frequency_divisions=8)
wdm_finer_frequency = transform(observed, "wdm", num_time_divisions=8)

for label, representation in (
    ("Time", observed),
    ("Frequency", spectrum),
    ("WDM: frequency selector", wdm_data),
    ("WDM: time selector", wdm_finer_frequency),
):
    print(f"{label:26s} shape={representation.channel_data['A'].shape}")
    if representation.wdm_grid is not None:
        print(f"  {representation.wdm_grid}")

assert wdm_data.channel_data["A"].shape == (9, 16)
assert wdm_finer_frequency.channel_data["A"].shape == (17, 8)

The two grids trade time resolution for frequency resolution. The WDM plots use
coefficient indices, and all representations show channel A. Inactive WDM edge
entries are masked in the plots; they are structural storage positions, not
missing observations or a statistical noise mask.

In [ ]:
figure, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
axes[0, 0].plot(sample_times_s, observed.channel_data["A"])
axes[0, 0].set(
    title="Observed channel A", xlabel="Time from start [s]", ylabel="Strain"
)

frequencies_hz = np.fft.rfftfreq(num_time_samples, sample_interval_s)
axes[0, 1].plot(frequencies_hz, np.abs(spectrum.channel_data["A"]))
axes[0, 1].set(
    title="Fourier coefficient magnitude", xlabel="Frequency [Hz]", ylabel="|dt · rFFT|"
)

coefficient_limit = max(
    float(np.max(np.abs(representation.channel_data["A"])))
    for representation in (wdm_data, wdm_finer_frequency)
)
for axis, representation in zip(axes[1], (wdm_data, wdm_finer_frequency), strict=True):
    grid = representation.wdm_grid
    visible_coefficients = np.ma.array(
        representation.channel_data["A"], mask=~grid.active_mask
    )
    mesh = axis.imshow(
        visible_coefficients,
        origin="lower",
        aspect="auto",
        interpolation="nearest",
        cmap="RdBu_r",
        vmin=-coefficient_limit,
        vmax=coefficient_limit,
    )
    axis.set(
        title=f"WDM: {grid.num_frequency_divisions} × {grid.num_time_divisions} divisions",
        xlabel="Time division index",
        ylabel="Frequency row index",
    )
    figure.colorbar(mesh, ax=axis, label="WDM coefficient")
plt.show()

## 3. Round trips, regridding, and independent copies

The same wrapper converts back to time or Fourier space. A WDM-to-WDM call can
select a different grid for the same underlying samples. Without new division
counts, it returns an independent copy on the existing grid.

In [ ]:
restored_from_frequency = transform(spectrum, "time")
restored_from_wdm = transform(wdm_data, "time")
regridded_wdm = transform(wdm_data, "wdm", num_time_divisions=8)
copied_wdm = transform(wdm_data, "wdm")

for label, restored in (
    ("Fourier round trip", restored_from_frequency),
    ("WDM round trip", restored_from_wdm),
):
    maximum_error = max(
        float(np.max(np.abs(restored.channel_data[name] - observed.channel_data[name])))
        for name in channel_names
    )
    print(f"{label}: maximum absolute error = {maximum_error:.3e}")
    for name in channel_names:
        np.testing.assert_allclose(
            restored.channel_data[name], observed.channel_data[name], atol=1e-12
        )
    assert restored.start_time_gps == observed.start_time_gps
    assert restored.num_time_samples == observed.num_time_samples

for name in channel_names:
    np.testing.assert_allclose(
        regridded_wdm.channel_data[name],
        wdm_finer_frequency.channel_data[name],
        atol=1e-12,
    )
    assert not np.shares_memory(
        copied_wdm.channel_data[name], wdm_data.channel_data[name]
    )
print("Regridding preserves the samples; copies own their arrays.")

## 4. Translate covariance with the same grid choice

A covariance that is independent at each native point can correlate different
points after translation. `TranslatedCovariance` retains those correlations and
provides exact `apply`, `solve`, `project`, and `quadratic_form` operations without
allocating a full dense observation covariance.

Match the covariance's target WDM grid to the data's grid. The covariance-weighted
quadratic form is invariant across representations. `log_determinant()` includes
the coordinate Jacobian and is not generally invariant.

In [ ]:
frequency_noise = transform(noise_covariance, "frequency")
wdm_noise = transform(
    noise_covariance,
    "wdm",
    num_frequency_divisions=wdm_data.wdm_grid.num_frequency_divisions,
)
assert isinstance(wdm_noise, TranslatedCovariance)

quadratic_forms = {
    "time": noise_covariance.quadratic_form(observed.channel_data),
    "frequency": frequency_noise.quadratic_form(spectrum.channel_data),
    "wdm": wdm_noise.quadratic_form(wdm_data.channel_data),
}
for domain, quadratic_form in quadratic_forms.items():
    print(f"{domain:10s}: quadratic form = {quadratic_form:.12f}")
np.testing.assert_allclose(
    list(quadratic_forms.values()), quadratic_forms["time"], rtol=1e-12
)

precision_weighted = wdm_noise.solve(wdm_data.channel_data)
recovered_data = wdm_noise.apply(precision_weighted)
for name in channel_names:
    np.testing.assert_allclose(
        recovered_data[name], wdm_data.channel_data[name], atol=1e-12
    )
print(f"Active real degrees of freedom: {wdm_noise.degrees_of_freedom}")

Native covariance's `active_mask` defines statistical exclusions. A translated
mask may span many coefficients, so a translated covariance exposes `project`
and has `active_mask=None`. `wdm_data.wdm_grid.active_mask` describes only the
WDM basis's structural positions; it cannot replace a statistical mask.

## 5. Translate a complete BlockResult

A `BlockResult` carries its signal, optional covariance, model parameters,
sampler state, and metadata. It deliberately has no observation-grid fields.
For a result containing signal arrays, pass its **source** `L1Data` as
`reference_data`. The covariance carries its own source grid.

The wrapper converts both estimates and independently copies the state
dictionaries without reinterpreting their contents.

In [ ]:
block_result = observed.block_result(
    tdi_signal_contribution=injected_signal,
    noise_covariance=noise_covariance,
    model_parameters={"amplitudes": np.array([1.0, 0.6])},
    sampler_state={"iteration": 0},
    metadata={"description": "Known injected signal for the translation demo"},
)
wdm_result = transform(
    block_result,
    "wdm",
    num_frequency_divisions=8,
    reference_data=observed,
)
restored_result = transform(wdm_result, "time", reference_data=wdm_data)

for name in channel_names:
    np.testing.assert_allclose(
        restored_result.tdi_signal_contribution[name], injected_signal[name], atol=1e-12
    )
np.testing.assert_array_equal(
    restored_result.model_parameters["amplitudes"],
    block_result.model_parameters["amplitudes"],
)
assert restored_result.sampler_state == block_result.sampler_state
assert restored_result.metadata == block_result.metadata
assert isinstance(restored_result.noise_covariance, DataCovariance)
np.testing.assert_array_equal(
    restored_result.noise_covariance.covariance_matrix,
    noise_covariance.covariance_matrix,
)
assert not np.shares_memory(
    wdm_result.model_parameters["amplitudes"],
    block_result.model_parameters["amplitudes"],
)
print("WDM signal shape:", wdm_result.tdi_signal_contribution["A"].shape)
print("WDM covariance representation:", wdm_result.noise_covariance.data_domain)
print("Restored parameters:", restored_result.model_parameters)

## 6. Let Wheel translate for each block

This demonstration block returns zero TDI signals and records the domain, shape,
and number of sampling calls. Fixed configuration stays on the instance; changing
state travels in `BlockResult`. No block calls `transform` or performs cross-block
subtraction.

The caller prepares a complete zero-signal result on each block's chosen grid
and passes it as the required `initial_block_result`. Registration validates and
stores that result without sampling. `sample` receives the previous result in
the block's chosen domain and returns its complete next result. A real model
would replace the zero signals with its estimates.

In [ ]:
@dataclass(frozen=True)
class DomainProbe:
    name: str
    expected_domain: str

    def _result(
        self, conditional_residual, noise_covariance, num_sample_calls
    ) -> BlockResult:
        assert conditional_residual.data_domain == self.expected_domain
        assert noise_covariance is not None
        assert noise_covariance.data_domain == self.expected_domain
        return conditional_residual.zero_block_result(
            sampler_state={"num_sample_calls": num_sample_calls},
            metadata={
                "input_domain": conditional_residual.data_domain,
                "input_shape": conditional_residual.channel_data["A"].shape,
            },
        )

    def sample(
        self, conditional_residual, noise_covariance, current_block_result, *, rng
    ):
        for name in conditional_residual.channel_names:
            assert (
                current_block_result.tdi_signal_contribution[name].shape
                == conditional_residual.channel_data[name].shape
            )
        return self._result(
            conditional_residual,
            noise_covariance,
            current_block_result.sampler_state["num_sample_calls"] + 1,
        )

In [ ]:
wheel = Wheel(observed, initial_noise_covariance=noise_covariance, random_seed=3)
for reference_data in (observed, spectrum, wdm_finer_frequency):
    domain = reference_data.data_domain
    initial = reference_data.zero_block_result(
        sampler_state={"num_sample_calls": 0},
        metadata={
            "input_domain": domain,
            "input_shape": reference_data.channel_data["A"].shape,
        },
    )
    options = {"num_time_divisions": 8} if domain == "wdm" else {}
    wheel.add(
        DomainProbe(f"{domain}_block", domain),
        initial_block_result=initial,
        data_domain=domain,
        **options,
    )
wheel.run(num_cycles=2)

for name, result in wheel.ledger.snapshot().items():
    print(
        f"{name:16s} receives {result.metadata['input_domain']:9s} "
        f"{str(result.metadata['input_shape']):10s}; "
        f"ledger signal shape {result.tdi_signal_contribution['A'].shape}; "
        f"sample calls {result.sampler_state['num_sample_calls']}"
    )
    assert result.sampler_state["num_sample_calls"] == 2
    assert result.tdi_signal_contribution["A"].shape == observed.channel_data["A"].shape

assert wheel.working_residual.data_domain == "time"
for name in channel_names:
    np.testing.assert_array_equal(
        wheel.working_residual.channel_data[name], observed.channel_data[name]
    )
print("Canonical domain:", wheel.working_residual.data_domain)
print("The zero-signal blocks leave the observations and full residual unchanged.")

Wheel stores accepted estimates on the original observation grid. Before each
call, it translates the conditional residual, current covariance, and the
block's previous signal into the registered representation. Returned estimates
are converted back before adoption. Parameters and sampler state retain their
block-local meaning.

These signal-only blocks omit covariance, so Wheel retains the starting noise
model. A block that owns the noise model must return its complete covariance on
every call. Omitting `data_domain` in `Wheel.add` preserves the existing input
representations rather than selecting a new one.

See the [domain translation guide](../docs/domain-translation.md) for conventions
and covariance semantics, or [the Python script](domain_translation.py) for a
compact command-line walkthrough.